# 🛒 Instamart Customer Intelligence & Support Copilot

---

## Project Overview

This notebook is the **single-document walkthrough** of the Instamart Customer Intelligence & Support Copilot project.  
Every cell below is executable and reproduces the exact outputs described in the project report.

### What this project builds

| Part | Theme | Key Output |
|------|-------|------------|
| **A** | Data Foundation | Clean CSVs · SQLite DB · EDA |
| **B** | Customer Intelligence | RFM segments · Lapse-risk model |
| **C** | Support Ticket Triage | LLM classifier · PII redaction · Token budgets |
| **D** | Policy Retrieval & Refund Agent | TF-IDF RAG · ReAct tool-calling agent |

### Design principles
- **Fully offline** — no external API keys, no network calls, no cloud costs.
- **Deterministic** — fixed random seed; identical outputs every run.
- **Self-contained** — all data is synthetic and generated on the fly.

---

## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Part A — Data Foundation](#2-part-a--data-foundation)
3. [Part B — Customer Intelligence](#3-part-b--customer-intelligence)
4. [Part C — Support Ticket Triage](#4-part-c--support-ticket-triage)
5. [Part D — Policy Retrieval & Refund Agent](#5-part-d--policy-retrieval--refund-agent)

---
## 1. Environment Setup

### 1.1 Project root & working directory

All project modules (`mock_llm`, `part_d`, etc.) live in the project root.  
Adding the root to `sys.path` lets every cell import them directly without installing the project as a package.  
Setting `os.chdir` makes every relative path (`data/raw/`, `outputs/`, etc.) resolve correctly regardless of where Jupyter was launched.

In [ ]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path(globals().get('__vsc_ipynb_file__',
                    Path.cwd() / 'instamart_analysis.ipynb')).resolve().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root :', PROJECT_ROOT)
print('Working dir  :', Path.cwd())

### 1.2 Import all shared libraries

| Library | Purpose |
|---------|--------|
| `pandas` | Tabular data loading, cleaning, aggregation |
| `numpy` | Numerical arrays, random sampling |
| `scikit-learn` | Clustering, classification, preprocessing, metrics |
| `joblib` | Model serialisation |
| `plotly.express` | Interactive charts |
| `sqlite3` | Embedded relational database (stdlib) |
| `mock_llm` | Offline deterministic LLM stand-in (project module) |

In [ ]:
import csv, json, math, random, re, sqlite3, time, warnings
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sklearn

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, silhouette_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print('pandas       :', pd.__version__)
print('numpy        :', np.__version__)
print('scikit-learn :', sklearn.__version__)

---
## 2. Part A — Data Foundation

Part A is the bedrock of the entire project.  
It answers: *What data do we have, is it clean, and what does it tell us at a glance?*

### Stages
```
generate_data.py  →  clean_data.py  →  validate_data.py  →  part_a_eda.py
     ↓                   ↓                   ↓                    ↓
 data/raw/           data/clean/        instamart.db        part_a_eda.txt
```

### 2.1 Synthetic Data Generation

We generate all data programmatically so the project is fully self-contained.  
A fixed seed (`random.Random(42)`) guarantees identical data on every run.

#### What gets generated

| File | Rows | Description |
|------|------|-------------|
| `customers.csv` | 1 200 | IDs, cities, signup dates, ages, genders |
| `products.csv` | 40 | Product names, categories, prices |
| `orders.csv` | ~12 400 raw | Orders with intentional errors and duplicates |
| `tickets.csv` | 164 | Support tickets across 4 categories |
| `policy_eval.csv` | 14 | Q&A pairs for policy retrieval evaluation |
| `agent_scenarios.csv` | 7 | Refund request scenarios for the agent |
| `policies/*.txt` | 5 | Plain-text policy documents |

#### Intentional data quality problems planted
- Impossible ages (e.g. 280, −4, 999)
- Mixed-case / padded city names (`chennai`, `MUMBAI`, ` Delhi `)
- Gender variants (`M`, `m`, `MALE`, `Female`, `f`)
- Negative delivery minutes (invalid)
- Unparseable order amounts (`N/A`, `error`)
- ~5% partial duplicates (same order, earlier `pending` record)
- ~1% exact duplicate rows

In [ ]:
# Run the data generator — writes data/raw/*.csv and data/policies/*.txt
exec(open('generate_data.py').read())

# Quick inventory
print('\nGenerated files:')
for f in sorted(Path('data/raw').iterdir()):
    lines = sum(1 for _ in open(f, encoding='utf-8')) - 1
    print(f'  {f.name:<30} {lines:>6} rows')

### 2.2 Data Cleaning

Raw data is messy by design. The cleaning script applies rule-based transformations to produce analysis-ready CSVs in `data/clean/`.

#### Cleaning rules applied

| Column | Raw problem | Fix applied |
|--------|------------|-------------|
| `city` | Mixed case, whitespace | `.str.strip().str.title()` |
| `age` | Out-of-range (< 0 or > 120) | Set to `NaN` |
| `gender` | Many variants | Map to `male` / `female` / `NaN` |
| `order_amount` | `'N/A'`, `'error'` strings | `pd.to_numeric(..., errors='coerce')` |
| `delivery_minutes` | Negative values | Set to `NaN` |
| Duplicate orders | Partial + exact duplicates | Keep row with **latest** `order_ts` per `order_id` |

In [ ]:
from pathlib import Path
import pandas as pd

RAW   = Path('data/raw')
CLEAN = Path('data/clean')
CLEAN.mkdir(parents=True, exist_ok=True)

# ── 1. Customers ────────────────────────────────────────────────────────
customers = pd.read_csv(RAW / 'customers.csv')
customers['city']   = customers['city'].astype('string').str.strip().str.title()
customers['age']    = pd.to_numeric(customers['age'], errors='coerce')
customers.loc[~customers['age'].between(0, 120), 'age'] = pd.NA
customers['gender'] = customers['gender'].astype('string').str.strip().str.lower()
gender_map = {'f':'female','female':'female','m':'male','male':'male'}
customers['gender'] = customers['gender'].map(gender_map).astype('string')
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')
customers = customers.drop_duplicates(subset='customer_id', keep='first')
customers.to_csv(CLEAN / 'customers.csv', index=False)

# ── 2. Products ─────────────────────────────────────────────────────────
products = pd.read_csv(RAW / 'products.csv')
products['price_inr'] = pd.to_numeric(products['price_inr'], errors='coerce')
products = products.drop_duplicates(subset='product_id', keep='first')
products.to_csv(CLEAN / 'products.csv', index=False)

# ── 3. Orders ───────────────────────────────────────────────────────────
orders = pd.read_csv(RAW / 'orders.csv', dtype=str)
orders = orders.drop_duplicates()   # remove exact duplicates first
orders['order_ts']         = pd.to_datetime(orders['order_ts'], errors='coerce')
orders['quantity']         = pd.to_numeric(orders['quantity'], errors='coerce')
orders['discount_pct']     = pd.to_numeric(orders['discount_pct'], errors='coerce')
orders['order_amount']     = pd.to_numeric(orders['order_amount'].str.strip(), errors='coerce')
orders['delivery_minutes'] = pd.to_numeric(orders['delivery_minutes'], errors='coerce')
orders.loc[orders['delivery_minutes'] < 0, 'delivery_minutes'] = pd.NA
orders = (
    orders
    .sort_values(['order_id','order_ts'], kind='stable', na_position='first')
    .drop_duplicates(subset='order_id', keep='last')
)
orders.to_csv(CLEAN / 'orders.csv', index=False)

print('=== CLEANING SUMMARY ===')
print(f'Customers  : {len(customers):>6}')
print(f'Products   : {len(products):>6}')
print(f'Orders     : {len(orders):>6}  (unique order IDs)')
print(f'Missing order amounts  : {orders["order_amount"].isna().sum()}')
print(f'Missing delivery times : {orders["delivery_minutes"].isna().sum()}')

### 2.3 Data Validation & SQLite Loading

Before any analysis we run **structural assertions** to confirm the clean data meets minimum quality standards, then load it into a local SQLite database so we can run SQL queries alongside pandas.

#### Checks performed
- Primary-key uniqueness for all three tables
- Referential integrity: every `order.customer_id` exists in `customers`
- No negative delivery times remain
- Quantity and discount values are in valid ranges
- SQLite row counts match CSV row counts

In [ ]:
import sqlite3

CLEAN   = Path('data/clean')
DB_PATH = Path('data/instamart.db')

customers = pd.read_csv(CLEAN / 'customers.csv')
products  = pd.read_csv(CLEAN / 'products.csv')
orders    = pd.read_csv(CLEAN / 'orders.csv')

assert customers['customer_id'].is_unique,  'Duplicate customer IDs'
assert products['product_id'].is_unique,    'Duplicate product IDs'
assert orders['order_id'].is_unique,        'Duplicate order IDs'
assert orders['customer_id'].isin(customers['customer_id']).all(), 'Orphan customer refs'
assert orders['product_id'].isin(products['product_id']).all(),    'Orphan product refs'
valid_del = orders['delivery_minutes'].dropna()
assert (valid_del >= 0).all(), 'Negative delivery times found'
print('PASS: all structural assertions cleared')

with sqlite3.connect(DB_PATH) as con:
    customers.to_sql('customers', con, if_exists='replace', index=False)
    products.to_sql('products',  con, if_exists='replace', index=False)
    orders.to_sql('orders',      con, if_exists='replace', index=False)

with sqlite3.connect(DB_PATH) as con:
    for tbl, df in [('customers',customers),('products',products),('orders',orders)]:
        n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
        assert n == len(df)
        print(f'  {tbl:<12}: {n:>6} rows  ✓')
print('\nSQLite database created at', DB_PATH)

### 2.4 SQL Exploratory Queries

SQL gives us a second independent view of the data — any discrepancy between a pandas result and its SQL equivalent reveals a bug.  

We run seven business queries covering order status, monthly revenue, top products, top customers, cancellation rates, delivery times, and category revenue.

In [ ]:
DB_PATH = Path('data/instamart.db')

def sql(query, label=''):
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql_query(query, con)

# Q1 — Order status distribution
print('── Q1: Order Status Counts ──')
display(sql('SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY 2 DESC'))

# Q2 — Monthly revenue
print('── Q2: Monthly Revenue ──')
display(sql("""
    SELECT substr(order_ts,1,7) AS month,
           COUNT(*) AS orders,
           ROUND(SUM(order_amount),2) AS revenue_inr
    FROM orders WHERE status='completed' GROUP BY 1 ORDER BY 1"""))

# Q3 — Top 10 products by revenue
print('── Q3: Top 10 Products ──')
display(sql("""
    SELECT p.product_name, p.category, COUNT(*) AS orders,
           ROUND(SUM(o.order_amount),2) AS revenue_inr
    FROM orders o JOIN products p ON o.product_id=p.product_id
    WHERE o.status='completed' GROUP BY p.product_id ORDER BY 4 DESC LIMIT 10"""))

# Q5 — Cancellation rate by city
print('── Q5: Cancellation Rate by City ──')
display(sql("""
    SELECT c.city, COUNT(*) AS total_orders,
           ROUND(100.0*SUM(CASE WHEN o.status='cancelled' THEN 1 ELSE 0 END)/COUNT(*),2)
               AS cancellation_rate_pct
    FROM orders o JOIN customers c ON o.customer_id=c.customer_id
    GROUP BY c.city ORDER BY 3 DESC"""))

# Q7 — Revenue by category
print('── Q7: Revenue by Category ──')
display(sql("""
    SELECT p.category, COUNT(*) AS completed_orders,
           ROUND(SUM(o.order_amount),2) AS revenue_inr
    FROM orders o JOIN products p ON o.product_id=p.product_id
    WHERE o.status='completed' GROUP BY p.category ORDER BY 3 DESC"""))

### 2.5 Exploratory Data Analysis (EDA)

We cross-validate pandas and SQL results to confirm consistency, then produce five core EDA charts.

**Cross-checks performed**
1. Order status counts — pandas vs SQL must be identical.
2. Monthly revenue — must agree within ₹0.01 rounding tolerance.

In [ ]:
orders = pd.read_csv('data/clean/orders.csv')
orders['order_ts'] = pd.to_datetime(orders['order_ts'], errors='coerce')
customers = pd.read_csv('data/clean/customers.csv')
products  = pd.read_csv('data/clean/products.csv')

# Cross-check 1
pandas_status = orders['status'].value_counts().sort_index()
with sqlite3.connect('data/instamart.db') as con:
    sql_status = pd.read_sql_query(
        'SELECT status, COUNT(*) AS count FROM orders GROUP BY status', con
    ).set_index('status')['count'].sort_index()
assert pandas_status.equals(sql_status)
print('PASS: order status counts match')

# Cross-check 2
completed = orders[orders['status'] == 'completed'].copy()
completed['order_month'] = completed['order_ts'].dt.strftime('%Y-%m')
pandas_monthly = completed.groupby('order_month')['order_amount'].sum(min_count=1).round(2)
with sqlite3.connect('data/instamart.db') as con:
    sql_monthly = pd.read_sql_query("""
        SELECT substr(order_ts,1,7) AS m, ROUND(SUM(order_amount),2) AS rev
        FROM orders WHERE status='completed' GROUP BY 1 ORDER BY 1""", con
    ).set_index('m')['rev']
assert (pandas_monthly - sql_monthly).abs().max() < 0.011
print('PASS: monthly revenue cross-check passed')

# Chart 1 — Order status
sc = orders['status'].value_counts().reset_index(); sc.columns = ['Status','Orders']
px.pie(sc, names='Status', values='Orders', title='Order Status Distribution', hole=0.45).show()

# Chart 2 — Monthly revenue
mr = completed.groupby('order_month')['order_amount'].sum().reset_index()
mr.columns = ['Month','Revenue (INR)']
px.bar(mr, x='Month', y='Revenue (INR)',
       title='Monthly Completed-Order Revenue', text_auto='.2s').show()

# Chart 3 — Revenue by category
cr = (completed.merge(products, on='product_id', how='left')
      .groupby('category')['order_amount'].sum().sort_values(ascending=False).reset_index())
cr.columns = ['Category','Revenue (INR)']
px.bar(cr, x='Category', y='Revenue (INR)', title='Revenue by Product Category',
       text_auto='.2s').show()

# Chart 4 — Order amount distribution
px.histogram(orders['order_amount'].dropna(), nbins=40,
             title='Order Amount Distribution (INR)',
             labels={'value':'INR','count':'Orders'}).show()

# Chart 5 — Delivery time distribution
fig5 = px.histogram(completed['delivery_minutes'].dropna(), nbins=30,
                    title='Delivery Time Distribution',
                    labels={'value':'Minutes','count':'Orders'})
fig5.add_vline(x=45, line_dash='dash', line_color='red', annotation_text='45-min SLA')
fig5.show()

# Save EDA report
Path('outputs').mkdir(exist_ok=True)
cancel_rate = (orders['status']=='cancelled').mean()*100
report = [
    'INSTAMART CUSTOMER INTELLIGENCE — PART A EDA', '='*48,
    f'Customers: {len(customers)}', f'Products: {len(products)}',
    f'Orders: {len(orders)}',
    f'Completed: {(orders["status"]=="completed").sum()}',
    f'Cancelled: {(orders["status"]=="cancelled").sum()}',
    f'Cancellation rate: {cancel_rate:.2f}%',
]
Path('outputs/part_a_eda.txt').write_text('\n'.join(report), encoding='utf-8')
print('EDA report saved → outputs/part_a_eda.txt')

---
## 3. Part B — Customer Intelligence

Part B answers two business questions:
1. **Who are my customers?** — RFM-based segmentation using K-Means clustering.
2. **Who is about to stop buying?** — A supervised lapse-risk classifier.

### Why these two questions matter
- Segmentation lets the marketing team craft targeted campaigns per group.
- The lapse-risk model lets the retention team intervene *before* a customer churns.

### Pipeline overview
```
Clean orders + customers
        │
        ▼
  RFM features (as of 2025-10-01)
        │
   ┌────┴────┐
   ▼         ▼
K-Means   Lapse cohort + leakage-safe features
   │             │
Segments   GridSearchCV (LR + RF)
   │             │
Profiles   Best model → lapse_model.joblib
```

### 3.1 RFM Feature Engineering

**Recency, Frequency, Monetary (RFM)** is a classic customer-value framework:

| Feature | Definition | Business meaning |
|---------|-----------|------------------|
| **Recency** | Days since last completed order (as of 2025-10-01) | Lower = more engaged |
| **Frequency** | Count of distinct completed orders | Higher = more loyal |
| **Monetary** | Total spend on completed orders (INR) | Higher = more valuable |

We compute RFM only for customers who placed **at least one completed order**.  
Customers with only cancelled orders are excluded — they were never actually charged.

In [ ]:
SNAPSHOT = pd.Timestamp('2025-10-01')
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)

orders_c = pd.read_csv('data/clean/orders.csv')
orders_c['order_ts'] = pd.to_datetime(orders_c['order_ts'], errors='coerce')
customers_c = pd.read_csv('data/clean/customers.csv')
customers_c['signup_date'] = pd.to_datetime(customers_c['signup_date'], errors='coerce')
products_c = pd.read_csv('data/clean/products.csv')

for col in ['quantity','discount_pct','order_amount','delivery_minutes']:
    orders_c[col] = pd.to_numeric(orders_c[col], errors='coerce')

completed_c = orders_c[orders_c['status'] == 'completed'].copy()

rfm = (
    completed_c.groupby('customer_id')
    .agg(last_order_ts=('order_ts','max'),
         frequency=('order_id','nunique'),
         monetary=('order_amount','sum'))
    .reset_index()
)
rfm['recency'] = (SNAPSHOT - rfm['last_order_ts']).dt.days
rfm = rfm[['customer_id','recency','frequency','monetary']]

print(f'Customers with ≥1 completed order: {len(rfm)}')
display(rfm[['recency','frequency','monetary']].describe().round(2))

### 3.2 K-Means Clustering (K = 2 … 8)

Before choosing K we evaluate two metrics across K = 2 to 8:

- **Inertia (WCSS):** always decreases as K increases — alone it is not a good criterion.
- **Silhouette score:** measures how similar a customer is to their own cluster vs neighbouring clusters. Range: −1 (bad) to +1 (perfect).

We look for the **elbow** in inertia and the **peak** in silhouette.  
**K = 4** is selected — strong silhouette and four segments that a marketing team can act on without confusion.

In [ ]:
rfm_cols = ['recency','frequency','monetary']
scaler   = StandardScaler()
X_rfm    = scaler.fit_transform(rfm[rfm_cols])

k_results = []
for k in range(2, 9):
    km     = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_rfm)
    k_results.append({'k':k, 'inertia':round(km.inertia_,2),
                      'silhouette':round(silhouette_score(X_rfm,labels),4)})

k_table = pd.DataFrame(k_results)
k_table.to_csv(OUT / 'kmeans_metrics.csv', index=False)
display(k_table)

fig = go.Figure()
fig.add_trace(go.Scatter(x=k_table['k'], y=k_table['inertia'],
                         name='Inertia', yaxis='y1', mode='lines+markers'))
fig.add_trace(go.Scatter(x=k_table['k'], y=k_table['silhouette'],
                         name='Silhouette', yaxis='y2', mode='lines+markers'))
fig.update_layout(title='K-Means Inertia & Silhouette (K=2…8)',
                  xaxis_title='K',
                  yaxis=dict(title='Inertia'),
                  yaxis2=dict(title='Silhouette', overlaying='y', side='right'))
fig.show()
print('\nSelected K = 4')

### 3.3 Segment Profiles & Naming

With K = 4 fit on scaled RFM, we assign each cluster a human-readable name based on its measured profile:

| Segment | Highest on | Business action |
|---------|-----------|----------------|
| Premium High-Value | Monetary | White-glove service, priority support |
| Loyal Frequent | Frequency | Loyalty rewards, subscription offers |
| Cooling / At-Risk | Recency (days ago) | Win-back campaigns |
| Steady Customers | Balanced | Nurture with cross-sell |

The naming is **data-driven** — we identify which cluster has the highest mean monetary, the next-highest frequency, the next-highest recency, and label the rest.

In [ ]:
FINAL_K = 4
kmeans = KMeans(n_clusters=FINAL_K, n_init=10, random_state=42)
rfm['segment_id'] = kmeans.fit_predict(X_rfm)

profiles = (
    rfm.groupby('segment_id')
    .agg(size=('customer_id','size'),
         mean_recency=('recency','mean'),
         mean_frequency=('frequency','mean'),
         mean_monetary=('monetary','mean'))
    .reset_index()
)

pb = profiles.set_index('segment_id')
premium_id   = int(pb['mean_monetary'].idxmax())
rem = [int(x) for x in profiles['segment_id'] if int(x) != premium_id]
freq_id = int(pb.loc[rem,'mean_frequency'].idxmax()); rem.remove(freq_id)
risk_id = int(pb.loc[rem,'mean_recency'].idxmax());   rem.remove(risk_id)
other_id = rem[0]

seg_names = {premium_id:'Premium High-Value', freq_id:'Loyal Frequent',
             risk_id:'Cooling / At-Risk', other_id:'Steady Customers'}

rfm['segment_name'] = rfm['segment_id'].map(seg_names)
rfm['is_premium']   = rfm['segment_id'].eq(premium_id)
profiles['segment_name'] = profiles['segment_id'].map(seg_names)
profiles['is_premium']   = profiles['segment_id'].eq(premium_id)
profiles.to_csv(OUT / 'segment_profiles.csv', index=False)

display(profiles.round(2))

fig = px.bar(profiles, x='segment_name',
             y=['mean_recency','mean_frequency','mean_monetary'],
             barmode='group', title='Mean RFM per Segment',
             labels={'value':'Mean','variable':'Metric','segment_name':'Segment'})
fig.show()

rfm[['customer_id','recency','frequency','monetary',
     'segment_id','segment_name','is_premium']].to_csv(OUT/'customer_segments.csv', index=False)
print('Saved: customer_segments.csv, segment_profiles.csv')

### 3.4 PCA Cluster Visualisation

PCA reduces three scaled RFM dimensions to **two principal components** for plotting.  
Well-separated clusters on this chart indicate genuinely distinct customer groups.  
Overlapping areas suggest customers who share similar behaviour patterns and may need more careful manual validation before targeted campaigns are run.

In [ ]:
pca    = PCA(n_components=2)
coords = pca.fit_transform(X_rfm)
rfm_v  = rfm.copy()
rfm_v['PC1'] = coords[:,0]; rfm_v['PC2'] = coords[:,1]

ev = pca.explained_variance_ratio_
print(f'Explained variance: PC1={ev[0]:.3f}, PC2={ev[1]:.3f}, Total={ev.sum():.3f}')

fig = px.scatter(rfm_v, x='PC1', y='PC2', color='segment_name', opacity=0.7,
                 hover_data=['customer_id','recency','frequency','monetary'],
                 title='Customer Segments — PCA Projection (RFM space)')
fig.show()

### 3.5 Lapse-Risk Model — Cohort & Feature Engineering

**Business question:** Which customers placed orders before September 2025 but did *not* order in September?  Can we predict this in advance?

#### Label definition
- **Cohort:** customers with ≥1 completed order before `2025-09-01`.
- **Lapsed = 1** if no completed order during September 2025.
- **Lapsed = 0** if they placed ≥1 completed order in September.

#### Leakage prevention
September orders are used **only to build the label**, never as model features.  
All features are computed strictly from data before `2025-09-01`.

#### Features used
| Type | Features |
|------|----------|
| Numeric | recency, frequency, monetary, avg_discount_pct, cancellation_rate, avg_delivery_minutes, age, tenure_days |
| Categorical | city, gender, favourite_category |

In [ ]:
CUTOFF  = pd.Timestamp('2025-09-01')
SEP_END = pd.Timestamp('2025-10-01')

orders_cat = orders_c.merge(products_c[['product_id','category']], on='product_id', how='left')
pre_cutoff    = orders_cat[orders_cat['order_ts'] < CUTOFF]
pre_completed = pre_cutoff[pre_cutoff['status'] == 'completed']
cohort_ids    = pre_completed['customer_id'].unique()

sep_completed = completed_c[(completed_c['order_ts']>=CUTOFF)&(completed_c['order_ts']<SEP_END)]
sep_ids       = set(sep_completed['customer_id'])

cohort = pd.DataFrame({'customer_id': cohort_ids})
cohort['lapsed'] = (~cohort['customer_id'].isin(sep_ids)).astype(int)

pre_rfm = (pre_completed.groupby('customer_id')
    .agg(last_order_ts=('order_ts','max'), frequency=('order_id','nunique'),
         monetary=('order_amount','sum'), avg_discount_pct=('discount_pct','mean'),
         avg_delivery_minutes=('delivery_minutes','mean')).reset_index())
pre_rfm['recency'] = (CUTOFF - pre_rfm['last_order_ts']).dt.days

st_cnt = (pre_cutoff
    .assign(is_canc=pre_cutoff['status'].eq('cancelled').astype(int))
    .groupby('customer_id')
    .agg(all_f=('order_id','nunique'), canc=('is_canc','sum')).reset_index())
st_cnt['cancellation_rate'] = st_cnt['canc'] / st_cnt['all_f'].replace(0, float('nan'))

fav = (pre_completed.groupby(['customer_id','category']).size().reset_index(name='n')
       .sort_values(['customer_id','n','category'], ascending=[True,False,True], kind='stable')
       .drop_duplicates('customer_id')[['customer_id','category']]
       .rename(columns={'category':'favourite_category'}))

features_df = (cohort
    .merge(pre_rfm, on='customer_id', how='left')
    .merge(st_cnt[['customer_id','cancellation_rate']], on='customer_id', how='left')
    .merge(fav, on='customer_id', how='left')
    .merge(customers_c[['customer_id','city','age','gender','signup_date']], on='customer_id', how='left'))
features_df['tenure_days'] = (CUTOFF - features_df['signup_date']).dt.days

print(f'Cohort size : {len(cohort)}')
print(f'Lapsed      : {cohort["lapsed"].sum()} ({cohort["lapsed"].mean()*100:.1f}%)')
display(features_df.head(3))

### 3.6 Model Training — GridSearchCV (Logistic Regression & Random Forest)

We train two models with 5-fold cross-validated `GridSearchCV`, then compare on a held-out test set.

#### Preprocessing (inside each model pipeline)
1. **Numeric:** median imputation → standard scaling
2. **Categorical:** most-frequent imputation → one-hot encoding

Preprocessing is fitted **only on training rows** inside each CV fold — this is the correct approach to avoid data leakage from validation folds.

#### Evaluation metric
We optimise for **F1 on the lapsed class** because:
- Missing a lapsing customer costs future orders (false negative = bad).
- Wrongly targeting a loyal customer wastes budget and may annoy them (false positive = bad).
- F1 balances both concerns.

In [ ]:
numeric_features     = ['recency','frequency','monetary','avg_discount_pct',
                         'cancellation_rate','avg_delivery_minutes','age','tenure_days']
categorical_features = ['city','gender','favourite_category']

X = features_df[numeric_features + categorical_features].copy()
y = features_df['lapsed'].copy()
cust_ids = features_df['customer_id'].copy()

X_tr, X_te, y_tr, y_te, id_tr, id_te = train_test_split(
    X, y, cust_ids, test_size=0.2, stratify=y, random_state=42)

def make_pre():
    return ColumnTransformer([
        ('num', Pipeline([('imp',SimpleImputer(strategy='median')),
                          ('sc', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                          ('ohe',OneHotEncoder(drop='first',handle_unknown='ignore'))]),
                          categorical_features)])

lr_pipe = Pipeline([('pre',make_pre()),('m',LogisticRegression(max_iter=1000))])
lr_grid = GridSearchCV(lr_pipe,{'m__C':[0.01,0.1,1,10,100]},cv=5,scoring='f1',n_jobs=-1)
lr_grid.fit(X_tr, y_tr)

rf_pipe = Pipeline([('pre',make_pre()),
                    ('m',RandomForestClassifier(n_estimators=100, random_state=42))])
rf_grid = GridSearchCV(rf_pipe,
    {'m__max_depth':[3,5,8],'m__min_samples_leaf':[5,10,20]},cv=5,scoring='f1',n_jobs=-1)
rf_grid.fit(X_tr, y_tr)

best_lr, best_rf = lr_grid.best_estimator_, rf_grid.best_estimator_
print('LR  best params:', lr_grid.best_params_, f'  CV F1={lr_grid.best_score_:.4f}')
print('RF  best params:', rf_grid.best_params_, f'  CV F1={rf_grid.best_score_:.4f}')

def report(model, label):
    pred = model.predict(X_te)
    print(f'\n{label}')
    cm = confusion_matrix(y_te, pred, labels=[0,1])
    print('Confusion matrix [[TN FP][FN TP]]:', cm.tolist())
    print(f'  Precision: {precision_score(y_te,pred,zero_division=0):.4f}')
    print(f'  Recall   : {recall_score(y_te,pred,zero_division=0):.4f}')
    print(f'  F1       : {f1_score(y_te,pred,zero_division=0):.4f}')
    return pred

lr_pred = report(best_lr, 'Logistic Regression — test set')
rf_pred = report(best_rf, 'Random Forest — test set')

rf_f1 = f1_score(y_te,rf_pred,zero_division=0)
lr_f1 = f1_score(y_te,lr_pred,zero_division=0)
sel_model, sel_name = (best_rf,'Random Forest') if rf_f1>=lr_f1 else (best_lr,'Logistic Regression')
print(f'\nSelected model: {sel_name}')

model_path = OUT / 'lapse_model.joblib'
joblib.dump(sel_model, model_path)
assert joblib.load(model_path).predict(X_te[:5]).tolist() == sel_model.predict(X_te[:5]).tolist()
print('PASS: model saved & reloaded →', model_path)

### 3.7 Feature Importance & Premium Retention List

**Feature importance** reveals which variables most influence the lapse prediction.  
This helps the product team decide where to invest (e.g. improving delivery speed if `avg_delivery_minutes` ranks high).

**Retention list:** top-10 premium customers in the test set with the highest predicted lapse probability — the people most worth a personal outreach call.

In [ ]:
rf_pre = best_rf.named_steps['pre']
rf_est = best_rf.named_steps['m']
importance = pd.DataFrame({
    'feature'   : rf_pre.get_feature_names_out(),
    'importance': rf_est.feature_importances_,
}).sort_values('importance', ascending=False)
importance.head(5).to_csv(OUT / 'rf_top_features.csv', index=False)

fig = px.bar(importance.head(12).sort_values('importance'),
             x='importance', y='feature', orientation='h',
             title='Top 12 Random Forest Feature Importances')
fig.show()

premium_ids = set(rfm.loc[rfm['is_premium'], 'customer_id'].astype(str))
test_res = pd.DataFrame({
    'customer_id'      : id_te.astype(str).to_numpy(),
    'actually_lapsed'  : y_te.to_numpy(),
    'lapse_probability': sel_model.predict_proba(X_te)[:,1],
})
retention = (test_res[test_res['customer_id'].isin(premium_ids)]
             .sort_values('lapse_probability', ascending=False)
             .head(10)[['customer_id','lapse_probability','actually_lapsed']])
retention.to_csv(OUT / 'retention_list.csv', index=False)
print('Top-10 premium customers at highest lapse risk:')
display(retention.round(4))

---
## 4. Part C — Support Ticket Triage

Part C automates the classification and routing of customer support tickets using an offline mock LLM.

### Why this matters
Manual ticket routing is slow, inconsistent, and doesn't scale.  
An automated triage pipeline can:
- Route fraud tickets to the security team immediately.
- Escalate premium customers to a dedicated desk within SLA.
- Flag empty or oversized tickets for human review.
- Demonstrate safe PII handling before any LLM call.

### Pipeline overview
```
164 tickets loaded
    │
    ├── 4 empty  ────────────────────────────────► skipped_empty
    ├── 4 long (> 400 tokens) ─────────────────► flagged_long
    └── 156 eligible
          │
          ▼
    Redact PII (emails, phones, order IDs)
          │
          ▼
    Call mock LLM  ◄─── retry on TransientLLMError (max 3, exp. backoff)
          │
          ├── Valid JSON + allowed category ───► classified_json
          ├── Plain text, single keyword ──────► classified_keyword
          └── Anything else ──────────────────► flagged_invalid_output
                    │
                    ▼
          Priority + routing
          ┌─────────────────────────────────────┐
          │ fraud          → HIGH · security_team│
          │ premium cust.  → HIGH · premium_desk │
          │ human statuses → NORMAL · human_review│
          │ others         → NORMAL · support_queue│
          └─────────────────────────────────────┘
```

### 4.1 PII Redaction

**Privacy-by-design:** every ticket is redacted *before* any model call.  
Three patterns are replaced with placeholder tokens:

| Pattern | Regex | Replacement |
|---------|-------|-------------|
| Email addresses | `[\w.+-]+@[\w-]+\.[\w.]+` | `[EMAIL]` |
| Indian mobile numbers | `\b[6-9]\d{9}\b` | `[PHONE]` |
| Order IDs | `ORD-\d{6}` | `[ORDER_ID]` |

An assertion checks that no pattern survives redaction.  
The original text is kept internally but never sent to the model.

In [ ]:
import mock_llm, re
from collections import Counter

EMAIL_RE = re.compile(r'[\w.+-]+@[\w-]+\.[\w.]+')
PHONE_RE = re.compile(r'\b[6-9]\d{9}\b')
ORDER_RE = re.compile(r'ORD-\d{6}')
SENSITIVE = (EMAIL_RE, PHONE_RE, ORDER_RE)

def redact(text):
    counts = {}
    text, counts['emails']    = EMAIL_RE.subn('[EMAIL]', text)
    text, counts['phones']    = PHONE_RE.subn('[PHONE]', text)
    text, counts['order_ids'] = ORDER_RE.subn('[ORDER_ID]', text)
    assert not any(p.search(text) for p in SENSITIVE), 'PII survived redaction!'
    return text, counts

tickets_raw = []
with open('data/raw/tickets.csv', encoding='utf-8-sig', newline='') as f:
    tickets_raw = list(csv.DictReader(f))

redaction_totals = Counter()
for t in tickets_raw:
    orig = t.get('text') or ''
    t['_redacted'], counts = redact(orig)
    redaction_totals.update(counts)

print(f'Total tickets : {len(tickets_raw)}')
print(f'Redactions    : {dict(redaction_totals)}')
print(f'Total PII hits: {sum(redaction_totals.values())}')

### 4.2 Classification Pipeline

We test two prompt strategies:

| Strategy | System prompt content | Expected accuracy |
|----------|-----------------------|------------------|
| **Zero-shot** | Category names only | ~85% |
| **Few-shot** | Category names + one example per class | 100% |

Each LLM response is validated:
- **Valid JSON** with an allowed category → `classified_json`
- **Plain text** with exactly one matching keyword → `classified_keyword`
- **Anything else** → `flagged_invalid_output`

On `TransientLLMError`, we retry up to 3 times with exponential backoff.

In [ ]:
MAX_INPUT_TOKENS = 400
MAX_RETRIES = 3; BASE_DELAY = 0.05
ALLOWED = {'billing','delivery','technical','fraud'}
CLASSIFIED = {'classified_json','classified_keyword'}
HUMAN_STATUSES = {'skipped_empty','flagged_long','flagged_invalid_output','budget_exhausted'}

ZERO_SYS = ('CLASSIFY support tickets. '
            'Allowed categories: billing, delivery, technical, fraud. '
            'Return JSON only: {"category": "...", "reason": "..."}')

FEW_SYS = ZERO_SYS + """
EXAMPLES:
Fraud example: {"category": "fraud", "reason": "unauthorized transaction"}
Billing example: {"category": "billing", "reason": "incorrect invoice amount"}
Delivery example: {"category": "delivery", "reason": "late parcel"}
"""

def validate_reply(content):
    try:
        parsed = json.loads(content)
    except (json.JSONDecodeError, TypeError):
        words = re.findall(r'[a-z]+', str(content).lower())
        matches = [w for w in words if w in ALLOWED]
        return ('classified_keyword', matches[0]) if len(matches)==1 else ('flagged_invalid_output','')
    if isinstance(parsed,dict) and parsed.get('category') in ALLOWED:
        return 'classified_json', parsed['category']
    return 'flagged_invalid_output', ''

def get_route(status, category, is_premium):
    if status in HUMAN_STATUSES: return 'NORMAL', 'human_review'
    if category == 'fraud':      return 'HIGH',   'security_team'
    if is_premium:               return 'HIGH',   'premium_desk'
    return 'NORMAL', 'support_queue'

seg_rows = []
with open('outputs/customer_segments.csv', encoding='utf-8-sig') as f:
    seg_rows = list(csv.DictReader(f))
segments = {r['customer_id']: r for r in seg_rows}

OUTPUT_COLS = ['ticket_id','customer_id','status','category',
               'segment_name','priority','route','tokens_used']

def run_pipeline(tickets, sys_prompt, token_budget=None):
    results = []; total_tokens = 0; retries = 0
    mock_llm.reset()
    for t in tickets:
        tid = t['ticket_id']; cid = t['customer_id']
        orig = t.get('text') or ''
        safe = t['_redacted']
        seg  = segments.get(cid, {})
        is_prem = str(seg.get('is_premium','')).strip().lower() in {'true','1','yes'}
        status = ''; category = ''; ticket_tokens = 0
        if not orig.strip():
            status = 'skipped_empty'
        elif mock_llm.est_tokens(orig) > MAX_INPUT_TOKENS:
            status = 'flagged_long'
        else:
            msgs = [{'role':'system','content':sys_prompt},{'role':'user','content':safe}]
            retries_used = 0
            while True:
                if token_budget and total_tokens >= token_budget:
                    status='budget_exhausted'; break
                try:
                    resp = mock_llm.chat(msgs)
                    used = int(resp.usage.total_tokens)
                    total_tokens += used; ticket_tokens += used
                    status, category = validate_reply(resp.choices[0].message.content)
                    break
                except mock_llm.TransientLLMError:
                    if retries_used >= MAX_RETRIES: status='flagged_invalid_output'; break
                    time.sleep(BASE_DELAY * (2**retries_used)); retries_used+=1; retries+=1
        priority, route = get_route(status, category, is_prem)
        results.append({'ticket_id':tid,'customer_id':cid,'status':status,
                         'category':category,'segment_name':seg.get('segment_name',''),
                         'priority':priority,'route':route,'tokens_used':ticket_tokens,
                         'true_category':t.get('true_category','')})
    return {'rows':results,'total_tokens':total_tokens,'retries':retries}

print('Pipeline functions defined. Ready to run.')

### 4.3 Zero-Shot vs Few-Shot Evaluation

We run both prompt variants without a token budget and compare accuracy.

**Expected results:**
- Zero-shot: ~85.7% accuracy on classified tickets
- Few-shot: 100% accuracy on classified tickets

The few-shot improvement comes from examples that help the mock LLM disambiguate fraud tickets, which zero-shot sometimes misclassifies as billing.

In [ ]:
zero = run_pipeline(tickets_raw, ZERO_SYS)
few  = run_pipeline(tickets_raw, FEW_SYS)

def print_metrics(label, run):
    rows = run['rows']
    classified = [r for r in rows if r['status'] in CLASSIFIED]
    correct = sum(r['category']==r['true_category'] for r in classified)
    acc = correct/len(classified) if classified else 0
    print(f'\n── {label} ──')
    print(f'  Classified tickets : {len(classified)}')
    print(f'  Correct            : {correct}/{len(classified)}')
    print(f'  Accuracy           : {acc:.1%}')
    print(f'  Total tokens       : {run["total_tokens"]}')
    print(f'  Retries recovered  : {run["retries"]}')
    print(f'  Status counts      : {dict(Counter(r["status"] for r in rows))}')
    return classified

z_cls = print_metrics('ZERO-SHOT', zero)
f_cls = print_metrics('FEW-SHOT',  few)

# Confusion matrix for zero-shot
labels = ['billing','delivery','technical','fraud']
matrix = Counter((r['true_category'],r['category']) for r in z_cls)
print('\nZero-shot confusion table (rows=true, cols=predicted):')
print('true\\pred  ' + '  '.join(f'{l:>9}' for l in labels))
for actual in labels:
    print(f'{actual:<10}' + '  '.join(f'{matrix[(actual,p)]:>9}' for p in labels))

# Category distribution chart
cat_df = pd.DataFrame({'category':[r['category'] for r in f_cls if r['category']]})
fig = px.pie(cat_df, names='category', title='Few-shot: Classified Ticket Categories', hole=0.4)
fig.show()

### 4.4 Token Budget Demonstration

In production, every LLM call has a cost. A token budget lets the platform guarantee spend limits per run.  

When `total_tokens >= budget`, the pipeline stops calling the LLM and marks remaining tickets as `budget_exhausted`.  

This demonstration shows the budget enforcement in action.

In [ ]:
# Try 6000 first; drop to 5000 if no tickets are exhausted
budget_demo = run_pipeline(tickets_raw, ZERO_SYS, token_budget=6000)
exhausted = sum(r['status']=='budget_exhausted' for r in budget_demo['rows'])
demo_budget = 6000
if exhausted == 0:
    demo_budget = 5000
    budget_demo = run_pipeline(tickets_raw, ZERO_SYS, token_budget=5000)
    exhausted = sum(r['status']=='budget_exhausted' for r in budget_demo['rows'])

print(f'Token budget used   : {demo_budget}')
print(f'Budget-exhausted    : {exhausted} tickets')
print(f'Tokens consumed     : {budget_demo["total_tokens"]}')

Path('outputs').mkdir(exist_ok=True)
budget_df = pd.DataFrame(budget_demo['rows'])
budget_df[[c for c in OUTPUT_COLS if c in budget_df.columns]].to_csv(
    'outputs/triage_budget_demo.csv', index=False)
print('Saved: outputs/triage_budget_demo.csv')

# Save final few-shot full run
final_run = run_pipeline(tickets_raw, FEW_SYS, token_budget=100_000)
final_df  = pd.DataFrame(final_run['rows'])
final_df[[c for c in OUTPUT_COLS if c in final_df.columns]].to_csv(
    'outputs/triage_results.csv', index=False)
print('Saved: outputs/triage_results.csv')
print(f'Final run tokens    : {final_run["total_tokens"]}')

---
## 5. Part D — Policy Retrieval & Refund Agent

Part D combines two AI capabilities:
1. **Policy RAG:** Retrieve the right policy text to answer a customer question.
2. **ReAct agent:** Use tools to verify facts and process a refund end-to-end.

### Why these matter
- Hardcoded rules break when policies change. A retrieval-based system reads the latest policy document automatically.
- A tool-calling agent can handle multi-step decisions (look up order → check policy → calculate refund → issue or escalate) without a human in the loop for the straightforward cases.

### 5.1 TF-IDF Policy Retrieval

#### How it works
1. Each policy `.txt` file is split into **40-word chunks** with a configurable word overlap.
2. All chunks are vectorised using `TfidfVectorizer`.
3. At query time, the question is vectorised the same way and **cosine similarity** ranks the chunks.
4. The top-3 chunks are included as context in the LLM prompt.

#### Why overlap matters
With **overlap = 0**, a sentence split across a chunk boundary loses context.  
With **overlap = 10**, each chunk shares 10 words with its neighbours, preserving evidence that would otherwise be lost.

#### TF-IDF explained
- **TF (Term Frequency):** how often a term appears in one chunk.
- **IDF (Inverse Document Frequency):** penalises terms common across many chunks.
- Result: rare, specific words (e.g. `escalate`, `1000`) get high weight; common words (e.g. `the`, `is`) are down-weighted automatically.

In [ ]:
POLICY_DIR = Path('data/policies')
EVAL_FILE  = Path('data/raw/policy_eval.csv')
CHUNK_SIZE = 40

def chunk_documents(overlap=10):
    if not 0 <= overlap < CHUNK_SIZE:
        raise ValueError('overlap must be 0 to 39')
    chunks = []; step = CHUNK_SIZE - overlap
    for path in sorted(POLICY_DIR.glob('*.txt')):
        words = path.read_text(encoding='utf-8').split()
        start = 0
        while start < len(words):
            sel = words[start:start+CHUNK_SIZE]
            if not sel: break
            chunks.append({'source': path.name, 'text': ' '.join(sel)})
            if start + CHUNK_SIZE >= len(words): break
            start += step
    return chunks

class PolicyRetriever:
    def __init__(self, overlap=10):
        self.chunks = chunk_documents(overlap)
        self.vectorizer = TfidfVectorizer()
        self.matrix = self.vectorizer.fit_transform([c['text'] for c in self.chunks])
    def retrieve(self, query, k=3):
        qv = self.vectorizer.transform([query])
        scores = cosine_similarity(qv, self.matrix).ravel()
        ranked = sorted(range(len(self.chunks)), key=lambda i:(-scores[i],i))
        return [{'source':self.chunks[i]['source'],'score':float(scores[i]),
                 'text':self.chunks[i]['text']} for i in ranked[:k]]

r10 = PolicyRetriever(overlap=10)
r0  = PolicyRetriever(overlap=0)
print(f'Chunks with overlap=10 : {len(r10.chunks)}')
print(f'Chunks with overlap=0  : {len(r0.chunks)}')

top3 = r10.retrieve('refund policy late delivery', k=3)
print('\nTop-3 for "refund policy late delivery":')
for item in top3:
    print(f'  {item["source"]}  score={item["score"]:.4f}')

### 5.2 Policy Retrieval Evaluation

We evaluate 14 questions (12 answerable, 2 unanswerable) against both retrievers.

#### Metrics
| Metric | Definition |
|--------|------------|
| **Retrieval hit** | Expected document appears in the top-3 retrieved chunks |
| **Fact hit** | Expected fact substring appears in the model's answer |
| **Fallback hit** | Unanswerable question triggers `I could not find...` response |

**Expected:**
- Overlap 10 → 12/12 retrieval hits, 12/12 fact hits, 2/2 fallbacks
- Overlap 0 → 12/12 retrieval hits, 10/12 fact hits (chunk boundaries cost 2 facts)

In [ ]:
import mock_llm

POLICY_SYS = ('You are a POLICY ASSISTANT. Answer only from the supplied policy '
              'context. If the supplied context does not contain the answer, respond '
              'with a sentence beginning exactly: I could not find. '
              'Do not invent policy facts.')

def answer(question, retriever):
    results = retriever.retrieve(question, k=3)
    context = '\n'.join(f'[{r["source"]}]\n{r["text"]}' for r in results)
    user_msg = f'CONTEXT:\n{context}\nQUESTION:\n{question}'
    resp = mock_llm.chat([{'role':'system','content':POLICY_SYS},
                          {'role':'user','content':user_msg}])
    return {'answer':resp.choices[0].message.content,
            'sources':[r['source'] for r in results],
            'tokens':resp.usage.total_tokens}

def evaluate(retriever, eval_rows):
    ret_hits=0; fact_hits=0; fallback_hits=0; details=[]
    for row in eval_rows:
        res = answer(row['question'], retriever)
        txt = res['answer']; edoc = row['expected_doc']; efact = row['expected_fact']
        if edoc != 'NONE':
            rh = edoc in res['sources']; fh = bool(efact) and efact in txt
            ret_hits += int(rh); fact_hits += int(fh)
        else:
            rh=False; fh=False; fallback_hits += int(txt.startswith('I could not find'))
        details.append({'question':row['question'],'expected_doc':edoc,
                         'sources':';'.join(res['sources']),
                         'answer':txt,'retrieval_hit':rh,'expected_fact':efact,
                         'fact_hit':fh})
    return {'ret_hits':ret_hits,'fact_hits':fact_hits,
            'fallback_hits':fallback_hits,'details':details}

with open(str(EVAL_FILE), encoding='utf-8-sig', newline='') as f:
    eval_rows = list(csv.DictReader(f))

ans_rows = [r for r in eval_rows if r['expected_doc']!='NONE']

rep10 = evaluate(r10, eval_rows)
rep0  = evaluate(r0,  eval_rows)

print(f'Overlap 10 → retrieval {rep10["ret_hits"]}/{len(ans_rows)}, '
      f'fact {rep10["fact_hits"]}/{len(ans_rows)}, '
      f'fallback {rep10["fallback_hits"]}/2')
print(f'Overlap  0 → retrieval {rep0["ret_hits"]}/{len(ans_rows)}, '
      f'fact {rep0["fact_hits"]}/{len(ans_rows)}, '
      f'fallback {rep0["fallback_hits"]}/2')

Path('outputs').mkdir(exist_ok=True)
pd.DataFrame(rep10['details']).to_csv('outputs/policy_eval_overlap10.csv', index=False)
pd.DataFrame(rep0['details']).to_csv('outputs/policy_eval_overlap0.csv',  index=False)
print('Saved: policy_eval_overlap10.csv, policy_eval_overlap0.csv')

### 5.3 Refund Agent — Tools & Architecture

The refund agent uses the **ReAct** pattern:  
> Reason → Act (call a tool) → Observe (read result) → Reason again → …

#### Available tools

| Tool | Input | Output | Purpose |
|------|-------|--------|---------|
| `get_order` | `order_id` | status, amount, delivery_minutes | Fetch order facts from SQLite |
| `search_policy` | `query` | top-3 policy chunks | Retrieve relevant policy text |
| `calculate_refund` | order_amount, delivery_minutes | refund_inr | Apply the 45-minute rule |
| `issue_refund` | order_id, refund_inr | confirmation | Record auto-approved refund |
| `escalate_to_human` | order_id, refund_inr | confirmation | Flag for human review |

#### Business rules encoded
- Only **completed** orders are eligible for refunds.
- Delivery must exceed **45 minutes** for a full refund.
- Refunds **≤ INR 1 000** are issued automatically.
- Refunds **> INR 1 000** must be escalated to a human lead.

#### Safety limits per run
| Limit | Default |
|-------|---------|
| Max LLM calls | 6 |
| Token budget | 6 000 |
| Time limit | 10 s |
| On breach | `FALLBACK_MAX_ITERATIONS / TOKEN_LIMIT / TIMEOUT` |

In [ ]:
import time
from datetime import datetime, timezone

DB_PATH      = Path('data/instamart.db')
LEDGER_FILE  = Path('outputs/refund_ledger.jsonl')
RUNS_FILE    = Path('outputs/agent_runs.jsonl')
STATE_DIR    = Path('state'); STATE_DIR.mkdir(exist_ok=True)
Path('outputs').mkdir(exist_ok=True)

ACTIVE_RETRIEVER = None

def get_order(order_id):
    with sqlite3.connect(DB_PATH) as con:
        con.row_factory = sqlite3.Row
        row = con.execute(
            'SELECT order_id,status,order_amount,delivery_minutes FROM orders WHERE order_id=?',
            (order_id,)).fetchone()
    if row is None: return {'error':'order not found'}
    return {k: row[k] for k in ('order_id','status','order_amount','delivery_minutes')}

def search_policy(query):
    return {'results': ACTIVE_RETRIEVER.retrieve(query, k=3)}

def calculate_refund(order_amount, delivery_minutes):
    if order_amount is None or delivery_minutes is None: return {'refund_inr':0.0}
    return {'refund_inr': round(float(order_amount),2) if float(delivery_minutes)>45 else 0.0}

def _record(tool_name, order_id, refund_inr):
    entry = {'timestamp':datetime.now(timezone.utc).isoformat(),
             'tool_name':tool_name,'order_id':order_id,'refund_inr':round(float(refund_inr),2)}
    with LEDGER_FILE.open('a', encoding='utf-8') as f: f.write(json.dumps(entry)+'\n')
    return {'refund_inr': round(float(refund_inr),2)}

def issue_refund(order_id, refund_inr):      return _record('issue_refund',order_id,refund_inr)
def escalate_to_human(order_id,refund_inr): return _record('escalate_to_human',order_id,refund_inr)

TOOL_MAP = {'get_order':get_order,'search_policy':search_policy,
            'calculate_refund':calculate_refund,'issue_refund':issue_refund,
            'escalate_to_human':escalate_to_human}

TOOLS_SPEC = [
    {'name':'get_order','description':'Fetch order status, amount and delivery time.',
     'parameters':{'type':'object','properties':{'order_id':{'type':'string'}},'required':['order_id']}},
    {'name':'search_policy','description':'Retrieve top policy chunks.',
     'parameters':{'type':'object','properties':{'query':{'type':'string'}},'required':['query']}},
    {'name':'calculate_refund','description':'Calculate eligible refund.',
     'parameters':{'type':'object','properties':
         {'order_amount':{'type':'number'},'delivery_minutes':{'type':['number','null']}},
         'required':['order_amount','delivery_minutes']}},
    {'name':'issue_refund','description':'Record auto-approved refund.',
     'parameters':{'type':'object','properties':
         {'order_id':{'type':'string'},'refund_inr':{'type':'number'}},'required':['order_id','refund_inr']}},
    {'name':'escalate_to_human','description':'Flag refund for human review.',
     'parameters':{'type':'object','properties':
         {'order_id':{'type':'string'},'refund_inr':{'type':'number'}},'required':['order_id','refund_inr']}},
]
print('Tool functions and specs defined.')

### 5.4 Agent Loop — Checkpointing & Safe Fallbacks

The agent loop runs until the LLM returns a final text answer (no more tool calls).  
A **JSON checkpoint is saved after every tool observation** so the run can be resumed after a crash without repeating any work.

#### Fallback conditions
| Condition | Status returned |
|-----------|----------------|
| Max LLM calls reached | `FALLBACK_MAX_ITERATIONS` |
| Token budget exceeded | `FALLBACK_TOKEN_LIMIT` |
| Wall-clock time exceeded | `FALLBACK_TIMEOUT` |
| Unknown tool requested | Error message in observation |

All three fallbacks hand the case to a human agent with a clear summary — the system never silently fails.

In [ ]:
class SimulatedCrash(RuntimeError): pass

def _fallback(status):
    msgs = {
        'FALLBACK_MAX_ITERATIONS': 'FALLBACK_MAX_ITERATIONS: Step limit reached. Human review required.',
        'FALLBACK_TOKEN_LIMIT':    'FALLBACK_TOKEN_LIMIT: Token budget reached. Human review required.',
        'FALLBACK_TIMEOUT':        'FALLBACK_TIMEOUT: Time limit reached. Human review required.',
    }
    return msgs.get(status, 'FALLBACK: Human review required.')

def _save_ckpt(state):
    STATE_DIR.mkdir(exist_ok=True)
    (STATE_DIR / f'{state["run_id"]}.json').write_text(
        json.dumps(state, indent=2, ensure_ascii=False), encoding='utf-8')

def _agent_status(text):
    return text.split(':',1)[0].strip() if ':' in text else 'FALLBACK'

def _run_state(state, max_iter=6, tok_limit=6000, time_limit=10, crash_after=None):
    started = time.monotonic()
    state.update({'max_iterations':max_iter,'token_limit':tok_limit,'time_limit_seconds':time_limit})
    while state['llm_calls'] < max_iter:
        if time.monotonic()-started >= time_limit:
            state['status']='FALLBACK_TIMEOUT'; state['final_answer']=_fallback(state['status']); return state
        resp = mock_llm.chat(state['messages'], tools=TOOLS_SPEC)
        state['llm_calls'] += 1
        state['tokens_used'] += getattr(resp.usage,'total_tokens',0)
        if state['tokens_used'] > tok_limit:
            state['status']='FALLBACK_TOKEN_LIMIT'; state['final_answer']=_fallback(state['status']); return state
        msg = resp.choices[0].message; tcs = getattr(msg,'tool_calls',None)
        if not tcs:
            state['final_answer'] = msg.content or 'Human review required.'
            state['status'] = _agent_status(state['final_answer']); return state
        calls = [{'id':c.id,'name':c.name,'arguments':c.arguments} for c in tcs]
        state['messages'].append({'role':'assistant','content':None,'tool_calls':calls})
        for call in calls:
            try: obs = TOOL_MAP[call['name']](**call['arguments'])
            except KeyError: obs = {'error':f'Unknown tool: {call["name"]}'}
            except Exception as e: obs = {'error':str(e)}
            state['messages'].append({'role':'tool','tool_call_id':call['id'],
                                       'name':call['name'],'content':json.dumps(obs)})
            state['steps'].append({'tool_name':call['name'],'arguments':call['arguments'],'observation':obs})
            state['tools_executed'].append(call['name'])
            state['step_number'] += 1
            _save_ckpt(state)
            if crash_after and len(state['tools_executed']) >= crash_after:
                raise SimulatedCrash(f'Crashed after {len(state["tools_executed"])} tools')
    state['status']='FALLBACK_MAX_ITERATIONS'; state['final_answer']=_fallback(state['status']); return state

def run_agent(user_msg, scenario_id, max_iter=6, tok_limit=6000, time_limit=10, run_id=None, crash_after=None):
    sid = ''.join(c if c.isalnum() or c in '_-' else '_' for c in (run_id or scenario_id))
    sys_p = ('You are an AGENT for customer support. Follow the refund policy. '
             'Check order status before considering a refund. Only completed orders '
             'can be refunded. Delivery must exceed 45 minutes for a full refund. '
             'Never automatically issue a refund above INR 1000; escalate to a human. '
             'Use the supplied tools to verify facts.')
    state = {'run_id':sid,'scenario_id':scenario_id,
             'messages':[{'role':'system','content':sys_p},{'role':'user','content':user_msg}],
             'tokens_used':0,'step_number':0,'tools_executed':[],'steps':[],'llm_calls':0}
    result = _run_state(state,max_iter,tok_limit,time_limit,crash_after)
    return {k: result.get(k) for k in
            ('scenario_id','status','final_answer','llm_calls','tokens_used','steps','tools_executed','run_id')}

def resume_agent(run_id):
    state = json.loads((STATE_DIR/f'{run_id}.json').read_text(encoding='utf-8'))
    return _run_state(state, state.get('max_iterations',6), state.get('token_limit',6000),
                      state.get('time_limit_seconds',10))

print('Agent functions defined.')

### 5.5 Running All Seven Scenarios

| Scenario | User message | Expected outcome |
|----------|-------------|------------------|
| S1, S2 | Late delivery, small refund (≤ INR 1 000) | `REFUND` |
| S3 | Late delivery, large refund (> INR 1 000) | `ESCALATE` |
| S4 | On-time delivery | `DENIED` |
| S5 | Cancelled order | `DENIED` |
| S6 | Unknown order ID | `NOT_FOUND` |
| S7 | Infinite-loop trigger | `FALLBACK` |

We also run three safety demonstrations:
- **Token-limit demo:** agent hits the budget after 1–2 LLM calls.
- **Timeout demo:** agent times out on the very first call.
- **Crash-recovery demo:** agent crashes after 2 tools, then resumes from checkpoint.

In [ ]:
ACTIVE_RETRIEVER = PolicyRetriever(overlap=10)

for path in (RUNS_FILE, LEDGER_FILE):
    path.parent.mkdir(exist_ok=True); path.write_text('', encoding='utf-8')

with open('data/raw/agent_scenarios.csv', encoding='utf-8', newline='') as f:
    scenarios = list(csv.DictReader(f))
sc_lookup = {r['scenario_id']:r for r in scenarios}

def append_run(result, sid=None):
    rec = {'scenario_id': sid or result.get('scenario_id'),
           'status': result.get('status','?'),
           'final_answer': result.get('final_answer',''),
           'llm_calls': result.get('llm_calls',0),
           'tokens_used': result.get('tokens_used',0),
           'steps': result.get('steps',[])}
    with RUNS_FILE.open('a', encoding='utf-8') as f: f.write(json.dumps(rec)+'\n')

print('\nPART D — REFUND AGENT SCENARIOS')
print('-'*70)
all_results = []
for sc in scenarios:
    mock_llm.reset()
    r = run_agent(sc['message'], sc['scenario_id'], max_iter=6, tok_limit=6000, time_limit=10)
    append_run(r)
    all_results.append(r)
    exp = sc['expected_outcome']
    act = 'FALLBACK' if r['status'].startswith('FALLBACK') else r['status']
    ok  = act == exp
    print(f"{sc['scenario_id']}: expected={exp:<12} actual={act:<12} "
          f"calls={r['llm_calls']} tokens={r['tokens_used']} {'PASS' if ok else 'CHECK'}")
    print(f"  → {r['final_answer']}")

# Token-limit demo
mock_llm.reset()
td = run_agent(sc_lookup['S1']['message'], 'S1_TOKEN_DEMO', tok_limit=200, run_id='S1_TOKEN_DEMO')
append_run(td); print(f'\nToken demo : {td["status"]} (tokens={td["tokens_used"]})')

# Timeout demo
mock_llm.reset()
tt = run_agent(sc_lookup['S1']['message'], 'S1_TIMEOUT_DEMO', time_limit=0, run_id='S1_TIMEOUT_DEMO')
append_run(tt); print(f'Timeout demo: {tt["status"]} (calls={tt["llm_calls"]})')

# Crash-recovery demo
crash_id  = 'S1_crash_resume'
crash_path = STATE_DIR / f'{crash_id}.json'
if crash_path.exists(): crash_path.unlink()
mock_llm.reset()
try:
    run_agent(sc_lookup['S1']['message'], 'S1_CRASH_RESUME', run_id=crash_id, crash_after=2)
except SimulatedCrash as e:
    print(f'\nCrash simulated: {e}')

crashed = json.loads(crash_path.read_text(encoding='utf-8'))
print('Tools before crash:', crashed['tools_executed'])

mock_llm.reset()
resumed = resume_agent(crash_id)
print(f'Resumed status : {resumed["status"]}')
print(f'All tools used : {resumed["tools_executed"]}')
print(f'Final answer   : {resumed["final_answer"]}')

expected_tools = ['get_order','search_policy','calculate_refund','issue_refund']
ok = resumed['status']=='REFUND' and resumed['tools_executed']==expected_tools
print(f'Crash-recovery : {"PASS" if ok else "CHECK"}')

print('\nRun log   →', RUNS_FILE)
print('Ledger    →', LEDGER_FILE)
print('Checkpoints in', STATE_DIR)

### 5.6 Scenario Results Summary & Visualisation

In [ ]:
runs_df = pd.read_json('outputs/agent_runs.jsonl', lines=True)
display(runs_df[['scenario_id','status','llm_calls','tokens_used','final_answer']].head(10))

main_runs = runs_df[runs_df['scenario_id'].isin([f'S{i}' for i in range(1,8)])]
if not main_runs.empty:
    fig = px.bar(main_runs, x='scenario_id', y='llm_calls', color='status',
                 title='LLM Calls per Scenario', labels={'scenario_id':'Scenario','llm_calls':'LLM calls'})
    fig.show()

    fig2 = px.bar(main_runs, x='scenario_id', y='tokens_used', color='status',
                  title='Token Usage per Scenario')
    fig2.show()

# Refund ledger
ledger_df = pd.read_json('outputs/refund_ledger.jsonl', lines=True)
if not ledger_df.empty:
    print('\nRefund / Escalation Ledger:')
    display(ledger_df)

---
## 6. Project Summary

All four parts have run successfully. Here is a checklist of outputs:

| Output file | Part | Status |
|-------------|------|--------|
| `data/instamart.db` | A | SQLite database |
| `outputs/part_a_eda.txt` | A | EDA summary |
| `outputs/customer_segments.csv` | B | RFM + cluster labels |
| `outputs/segment_profiles.csv` | B | Mean RFM per segment |
| `outputs/kmeans_metrics.csv` | B | Inertia + silhouette K=2…8 |
| `outputs/lapse_model.joblib` | B | Serialised classifier |
| `outputs/rf_top_features.csv` | B | Top-5 feature importances |
| `outputs/retention_list.csv` | B | Premium lapse-risk list |
| `outputs/triage_results.csv` | C | Few-shot ticket classifications |
| `outputs/triage_budget_demo.csv` | C | Token-budget demo |
| `outputs/policy_eval_overlap10.csv` | D | Retrieval eval (overlap=10) |
| `outputs/policy_eval_overlap0.csv` | D | Retrieval eval (overlap=0) |
| `outputs/agent_runs.jsonl` | D | Per-scenario agent records |
| `outputs/refund_ledger.jsonl` | D | Refund audit log |
| `state/*.json` | D | Agent checkpoints |

In [ ]:
print('=== FINAL OUTPUT INVENTORY ===')
output_files = [
    'data/instamart.db',
    'outputs/part_a_eda.txt',
    'outputs/customer_segments.csv',
    'outputs/segment_profiles.csv',
    'outputs/kmeans_metrics.csv',
    'outputs/lapse_model.joblib',
    'outputs/rf_top_features.csv',
    'outputs/retention_list.csv',
    'outputs/triage_results.csv',
    'outputs/triage_budget_demo.csv',
    'outputs/policy_eval_overlap10.csv',
    'outputs/policy_eval_overlap0.csv',
    'outputs/agent_runs.jsonl',
    'outputs/refund_ledger.jsonl',
]
for fp in output_files:
    p = Path(fp)
    status = f'OK  ({p.stat().st_size:>8,} bytes)' if p.exists() else 'MISSING'
    print(f'  {fp:<45} {status}')

ckpts = list(Path('state').glob('*.json'))
print(f'  state/*.json                              OK  ({len(ckpts)} checkpoint files)')
print('\n✅ All outputs verified. Notebook complete.')